In [ ]:
"""
Script for traversing Wikidata items to extract specific properties such as genres (P136) and country of origin (P495).
"""

import requests
import time
from typing import Dict, List, Optional, Tuple
import json
import tqdm


USER_AGENT = "WikidataImageDownloader/1.0 (mailto:<your_email@example.com>)"


class WikidataSearcher:
    def __init__(self, delay: float = 0.5):
        self.search_url = "https://www.wikidata.org/w/api.php"
        self.delay = delay
        self.last_request_time = 0
    
    def _throttle_request(self):

        current_time = time.time()
        time_since_last = current_time - self.last_request_time
        if time_since_last < self.delay:
            time.sleep(self.delay - time_since_last)
        self.last_request_time = time.time()
    
    def _make_request(self, params: Dict) -> Dict:
        
        headers = {
            'User-Agent': USER_AGENT
        }
        
        self._throttle_request()
        
        try:
            response = requests.get(self.search_url, params=params, headers=headers, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"Request error: {e}")
            if hasattr(e, 'response') and e.response is not None:
                print(f"Status code: {e.response.status_code}")
            return {}
    
    def gen_queries(self, original_query: str) -> List[str]:
       
        queries = [original_query]
        words = original_query.split()
        
        if len(words) > 1:
           
            for i in range(len(words), 0, -1):
                if i < len(words):  
                    queries.append(" ".join(words[:i]))
            
            
            common_words = ['portrait', 'of', 'by',
                            'the', 'a', 'an', 'painting', 'by']
            filtered_words = [word for word in words if word.lower() not in common_words]
            if filtered_words and len(filtered_words) != len(words):
                queries.append(" ".join(filtered_words))
            
            if len(words) >= 3:
                queries.append(" ".join(words[-2:]))  
                queries.append(words[-1])  # 
        unique_queries = []
        for query in queries:
            if query not in unique_queries:
                unique_queries.append(query)
        
        return unique_queries
    
    def search_entity(self, query: str) -> Optional[Dict]:
        """
        Search for an entity using various query combinations
        
        :param query: str - The search query
        """
        search_queries = self.gen_queries(query)
        
        for search_query in search_queries:
            print(f"Trying search: '{search_query}'")
            
            params = {
                'action': 'wbsearchentities',
                'format': 'json',
                'language': 'en',
                'search': search_query,
                'limit': 5
            }
            
            data = self._make_request(params)
            results = data.get('search', [])
            
            if results:
                first_result = results[0]
                print(f"Found: {first_result.get('label', 'Unknown')} ({first_result.get('id')})")
                return first_result
        
        print("No results found with any search variation")
        return None
    
    def get_entity_properties(self, qid: str) -> Dict:
        """
        Get entities from wikidata
        """
        params = {
            'action': 'wbgetentities',
            'format': 'json',
            'ids': qid,
            'languages': 'en'
        }
        
        data = self._make_request(params)
        return data.get('entities', {}).get(qid, {})
    
    def extract_property_info(self, entity_data: Dict, property_id: str, property_name: str) -> List[Tuple[str, str]]:
        """
        Extract property information (name and QID) from entity data
        
        """
        properties = []
        claims = entity_data.get('claims', {})
        property_claims = claims.get(property_id, [])
        
        for claim in property_claims:
            try:
                prop_qid = claim['mainsnak']['datavalue']['value']['id']
                prop_info = self.get_entity_properties(prop_qid)
                prop_name = prop_info.get('labels', {}).get('en', {}).get('value', 'Unknown')
                properties.append((prop_name, prop_qid))
            except (KeyError, TypeError):
                continue
        
        if properties:
            print(f"Found {property_name}(s): {[f'{name} ({qid})' for name, qid in properties]}")
        else:
            print(f"No {property_name} found")
            
        return properties
    
    def search_and_get_info(self, query: str) -> Optional[Dict]:
        """
        Search for query and get genres (P136) and country of origin (P495) and QID of the entity
        """
        print(f"Searching for: '{query}'\n")
        
        search_result = self.search_entity(query)
        if not search_result:
            return None
        
        qid = search_result.get('id')
        entity_label = search_result.get('label', 'Unknown')
        entity_description = search_result.get('description', 'No description')
        
        print(f"Entity Information:")
        print(f"QID: {qid}")
        print(f"Label: {entity_label}")
        print(f"Description: {entity_description}")
        
        print(f"Getting properties for {qid}...")
        entity_data = self.get_entity_properties(qid)
        
        if not entity_data:
            print("Could not retrieve entity properties")
            return {
                'qid': qid,
                'label': entity_label,
                'description': entity_description,
                'country_of_origin': [], 
                'instance_of': []
            }

        instance_of = self.extract_property_info(entity_data, 'P31', 'instance of')
        
        country_of_origin = self.extract_property_info(entity_data, 'P495', 'country of origin')
        
        
        result = {
            'qid': qid,
            'label': entity_label,
            'instance_of': instance_of[0][1] if len(instance_of) > 0 and instance_of[0][1] else "nan",
            'description': entity_description if entity_description else "nan",
            'country_of_origin': country_of_origin[0][1] if len(country_of_origin) > 0 and country_of_origin[0][1] else "nan",
        }
        
        return result


input_file = 'raw_muse_data.json' 

muse_data= []

def load_from_file(filename):
  
    with open(filename, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data


muse_data= load_from_file(input_file)

results = []
queries = []
for entity in muse_data: 
    title = entity['title']
    author = entity['artistName']
    query = f"{title} by {author}"
    queries.append(query)
    

path = "muse_no_qids_cube_mt.json"

processed_data= load_from_file(path)


searcher = WikidataSearcher()

for i, query in tqdm.tqdm(enumerate(queries), total=len(queries)):
    result = searcher.search_and_get_info(query)
    
    processed_data[i]['id'] = result['qid'] if result else "nan"
    processed_data[i]['P31'] = result.get('instance_of', "nan") if result else "nan"
    processed_data[i]['P495'] = result.get('country_of_origin', "nan") if result else "nan"
    
    processed_data[i]['P17'] = "nan"
    print(f"Processed {i+1}/{len(queries)}: {query} -> {result['qid'] if result else 'No result'}")
    

  0%|          | 0/300 [00:00<?, ?it/s]

Searching for: 'Portrait of Fritza Riedler by Klimt Gustav'

Trying search: 'Portrait of Fritza Riedler by Klimt Gustav'
Trying search: 'Portrait of Fritza Riedler by Klimt'
Trying search: 'Portrait of Fritza Riedler by'
Trying search: 'Portrait of Fritza Riedler'
Found: Fritza Riedler (Q28001663)
Entity Information:
QID: Q28001663
Label: Fritza Riedler
Description: painting by Gustav Klimt
Getting properties for Q28001663...
Found instance of(s): ['painting (Q3305213)']


  0%|          | 1/300 [00:03<19:05,  3.83s/it]

Found country of origin(s): ['Austria (Q40)']
Processed 1/300: Portrait of Fritza Riedler by Klimt Gustav -> Q28001663
Searching for: 'Alice by Modigliani Amedeo '

Trying search: 'Alice by Modigliani Amedeo '
Trying search: 'Alice by Modigliani'
Trying search: 'Alice by'
Found: Alice Byam (Q102375965)
Entity Information:
QID: Q102375965
Label: Alice Byam
Description: owner of Eliots estate on Antigua, died by 29 Feb 1828 (date will proven)
Getting properties for Q102375965...


  1%|          | 2/300 [00:06<15:19,  3.09s/it]

...

Found instance of(s): ['human (Q5)']
No country of origin found
Processed 298/300: Liza Minnelli by Andy Warhol -> Q14441
Searching for: 'Symphony in White no.1: The White Girl Portrait of Joanna Hiffernan by James McNeill Whistler'

Trying search: 'Symphony in White no.1: The White Girl Portrait of Joanna Hiffernan by James McNeill Whistler'
Trying search: 'Symphony in White no.1: The White Girl Portrait of Joanna Hiffernan by James McNeill'
Trying search: 'Symphony in White no.1: The White Girl Portrait of Joanna Hiffernan by James'
Trying search: 'Symphony in White no.1: The White Girl Portrait of Joanna Hiffernan by'
Trying search: 'Symphony in White no.1: The White Girl Portrait of Joanna Hiffernan'
Trying search: 'Symphony in White no.1: The White Girl Portrait of Joanna'
Trying search: 'Symphony in White no.1: The White Girl Portrait of'
Trying search: 'Symphony in White no.1: The White Girl Portrait'
Trying search: 'Symphony in White no.1: The White Girl'
Trying search: 'Sympho

100%|█████████▉| 299/300 [19:03<00:05,  5.45s/it]

Found country of origin(s): ['United States (Q30)']
Processed 299/300: Symphony in White no.1: The White Girl Portrait of Joanna Hiffernan by James McNeill Whistler -> Q7661687
Searching for: 'Color Study: Squares with Concentric Circles by Wassily Kandinsky'

Trying search: 'Color Study: Squares with Concentric Circles by Wassily Kandinsky'
Trying search: 'Color Study: Squares with Concentric Circles by Wassily'
Trying search: 'Color Study: Squares with Concentric Circles by'
Trying search: 'Color Study: Squares with Concentric Circles'
Trying search: 'Color Study: Squares with Concentric'
Trying search: 'Color Study: Squares with'
Trying search: 'Color Study: Squares'
Trying search: 'Color Study:'
Trying search: 'Color'
Found: color (Q22006653)
Entity Information:
QID: Q22006653
Label: color
Description: filmed or drawn in color, the opposite of black-and-white
Getting properties for Q22006653...


100%|██████████| 300/300 [19:09<00:00,  3.83s/it]

Found instance of(s): ['color scheme (Q859170)', 'cinematic technique (Q1001378)', 'color (Q1075)']
No country of origin found
Processed 300/300: Color Study: Squares with Concentric Circles by Wassily Kandinsky -> Q22006653


In [5]:
processed_data[0]

{'P31': 'Q3305213',
 'P279': '[]',
 'P495': 'Q40',
 'P17': 'nan',
 'P2012': ' ',
 'P361': '[]',
 'id': 'Q28001663',
 'name': 'Portrait of Fritza Riedler',
 'country': 'Austria',
 'domain': 'art',
 'prompt': 'A painting of Portrait of Fritza Riedler by Klimt Gustav from Golden phase period in Art Nouveau (Modern) style, realistic'}

In [6]:
# save processed data
path = "/mnt/rds/CUBE/thirdparty/Processed/muse_data_for_cube_mt_with_qids.json"
with open(path, 'w', encoding='utf-8') as f:
    json.dump(processed_data, f, ensure_ascii=False, indent=4)
